# PROMPT 3: Document Chunking

**Amaç:** 13,609 QA record'unu 300-token chunks'a bölmek

## Pipeline
```
Raw Records (13.6k)  ← Prompt 2 çıktısı
     ↓
Token count (Kaç token?)
     ↓
Sliding window (overlap=50)
     ↓
Extract chunks with metadata
     ↓
Chunks (45-50k)  → Prompt 4 (Dense Retrieval) için
```

## Temel Kavramlar

### Token nedir?
> Bir kelime veya kelime parçası = 1 token
> - "Türk" = 1 token
> - "Ceza" = 1 token  
> - "Kanunu" = 1 token
> - "Toplam: Türk Ceza Kanunu" = 4 token

### Neden 300 token?
- Embedding model'leri (~512 token limit) sığdırmak için
- Türkçe metin: 1 token ≈ 4 karakter
- 300 token ≈ 1200 karakter (3-4 paragraf)

### Overlap nedir?
- **Problem:** Chunks arasında context kaybolur
- **Solution:** Overlap = sonraki chunk'a kadar olan context
- **Benefit:** LLM'nin chunks arası koşulturmayı anlaması

```
Overlap=50 ile:
[====Chunk 1====]  ← Token 0-300
         [====Chunk 2====]  ← Token 250-550 (50 token overlap)
                  [====Chunk 3====]  ← Token 500-800
```

In [ ]:
import sys
import json
import logging
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Config
DATA_DIR = Path.cwd().parent / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
CHUNKED_DIR = DATA_DIR / 'chunked'
CHUNKED_DIR.mkdir(exist_ok=True)

print(f"✅ Paths:")
print(f"   Processed dir: {PROCESSED_DIR}")
print(f"   Chunked dir: {CHUNKED_DIR}")

## Cell 2: DocumentChunker İmport ve Kontrol

DocumentChunker sınıfını import edelim.

In [ ]:
from ingestion.chunker import DocumentChunker, setup_logging, ChunkStatistics

# Initialize chunker
chunker = DocumentChunker(
    chunk_size=300,        # 300 token per chunk
    overlap_size=50,       # 50 token overlap
    min_chunk_size=20,     # Minimum 20 token 
    tokenizer_name="distilbert-base-multilingual-cased"  # Turkish support
)

print(f"✅ DocumentChunker initialized:")
print(f"   Chunk size: {chunker.chunk_size} tokens")
print(f"   Overlap size: {chunker.overlap_size} tokens")
print(f"   Step size: {chunker.step_size} tokens")
print(f"   Min chunk size: {chunker.min_chunk_size} tokens")

## Cell 3: Tokenization Demo

Bir örnek metin üzerinde tokenization'ı gösterelim.

In [ ]:
# Example Turkish text
sample_text = """Türk Ceza Kanunu Madde 81: Kasten adam öldürme suçunda ceza, 
ağırlaştırıcı sebepler mevcudiyeti halinde, müebbet hapis cezası ile karşılanır. 
Süresi belirlenmesi hâkimin takdirine bağlı olup, ağırlaştırıcı sebeplerin mevcudiyeti 
halinde, hapis cezası artırılır."""

# Count tokens
token_count = chunker.count_tokens(sample_text)
print(f"Sample text:")
print(sample_text)
print(f"\n📊 Token analysis:")
print(f"   Character count: {len(sample_text)}")
print(f"   Token count: {token_count}")
print(f"   Avg chars/token: {len(sample_text)/token_count:.1f}")

## Cell 4: Sliding Window Demo

Uzun bir metin için sliding window'u gösterelim.

In [ ]:
# Create mock token IDs (simulate 500 tokens)
mock_tokens = list(range(500))  # [0, 1, 2, ..., 499]

# Create sliding windows
windows = chunker.create_sliding_windows(mock_tokens)

print(f"📊 Sliding Window Analysis (500 tokens):")
print(f"   Chunk size: {chunker.chunk_size}")
print(f"   Overlap: {chunker.overlap_size}")
print(f"   Step: {chunker.step_size}")
print(f"\n   Windows:")

for idx, (start, end) in enumerate(windows, 1):
    chunk_len = end - start
    print(f"   Chunk {idx}: tokens[{start}:{end}]  ({chunk_len} tokens)")

print(f"\n   Total chunks: {len(windows)}")
print(f"   Total coverage: {len(mock_tokens)} tokens")

## Cell 5: Chunking a Single Record

Sample record'u chunk'lama gösterelim.

In [ ]:
# Sample QA record (simulating data/processed/ file)
sample_record = {
    "id": "550e8400-e29b-41d4-a716-446655440000",
    "question": "Kasten adam öldürme suçu nedir?",
    "answer": """Türk Ceza Kanunu Madde 81: Kasten adam öldürme suçunda ceza, 
    ağırlaştırıcı sebepler mevcudiyeti halinde, müebbet hapis cezası ile karşılanır. 
    Süresi belirlenmesi hâkimin takdirine bağlı olup, ağırlaştırıcı sebeplerin mevcudiyeti 
    halinde, hapis cezası artırılır. Kasten adam öldürme suçunun taksiri şekli için ağırlaştırıcı 
    sebepler mevcudiyeti halinde, müebbet hapis cezası verilir.""",
    "source": "turkish_law_dataset",
    "category": "Ceza Hukuku",
    "law_name": "Türk Ceza Kanunu",
    "article_no": "81",
    "section": "Kişilere Karşı Suçlar",
    "citation": ""
}

# Chunk the record
chunks = chunker.chunk_record(sample_record, sample_record['id'])

print(f"📋 Record:")
print(f"   Question: {sample_record['question']}")
print(f"   Answer length: {len(sample_record['answer'])} chars")
print(f"\n📦 Chunks generated: {len(chunks)}")
print()

for chunk in chunks:
    print(f"Chunk {chunk['chunk_position']}/{chunk['total_chunks']}:")
    print(f"  ID: {chunk['chunk_id']}")
    print(f"  Tokens: {chunk['chunk_length_tokens']}")
    print(f"  Text preview: {chunk['chunk_text'][:100]}...")
    print(f"  Law: {chunk['law_name']} Madde {chunk['article_no']}")
    print()

## Cell 6: Load Processed Data

data/processed/ directory'deki tüm JSONL dosyalarını yükleyelim.

In [ ]:
# Load all processed files
processed_files = list(PROCESSED_DIR.glob('*.jsonl'))

print(f"📂 Processed files found: {len(processed_files)}")
print()

all_records = []

for file_path in processed_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f]
    
    all_records.extend(records)
    print(f"✅ {file_path.name}: {len(records)} records")

print(f"\n📊 Total records loaded: {len(all_records)}")

## Cell 7: Process All Records (Main Chunking)

Tüm records'ları chunk'la ve çıktı dosyasına yaz.

In [ ]:
# Main chunking process
start_time = datetime.now()
output_file = CHUNKED_DIR / 'turkish_law_chunked.jsonl'

total_records = 0
total_chunks = 0
total_tokens = 0
error_count = 0

# Write chunks to output file
with open(output_file, 'w', encoding='utf-8') as outf:
    for record in tqdm(all_records, desc="Chunking records", total=len(all_records)):
        try:
            record_id = record.get('id', f'record-{total_records}')
            chunks = chunker.chunk_record(record, record_id)
            
            for chunk in chunks:
                outf.write(json.dumps(chunk, ensure_ascii=False) + '\n')
                total_chunks += 1
                total_tokens += chunk['chunk_length_tokens']
            
            total_records += 1
        
        except Exception as e:
            print(f"Error processing record {record.get('id', '?')}: {e}")
            error_count += 1

end_time = datetime.now()
processing_time = (end_time - start_time).total_seconds()

print(f"\n✅ Chunking complete!")
print(f"\n📊 Statistics:")
print(f"   Records processed: {total_records:,}")
print(f"   Chunks created: {total_chunks:,}")
print(f"   Total tokens: {total_tokens:,}")
print(f"   Avg tokens/chunk: {total_tokens/total_chunks if total_chunks > 0 else 0:.1f}")
print(f"   Chunks/record: {total_chunks/total_records if total_records > 0 else 0:.2f}")
print(f"   Processing time: {processing_time:.2f}s")
print(f"   Speed: {total_records/processing_time if processing_time > 0 else 0:.0f} records/sec")
print(f"   Errors: {error_count}")
print(f"\n   Output file: {output_file}")
print(f"   Output size: {output_file.stat().st_size / (1024*1024):.2f} MB")

## Cell 8: Verify Output

Çıktı dosyasını kontrol edelim.

In [ ]:
# Read and verify output chunks
chunks_df_list = []

with open(output_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        chunk = json.loads(line)
        chunks_df_list.append({
            'chunk_id': chunk['chunk_id'],
            'source_record_id': chunk['source_record_id'],
            'chunk_tokens': chunk['chunk_length_tokens'],
            'law_name': chunk['law_name'],
            'article_no': chunk['article_no'],
            'source': chunk['source'],
        })
        
        if i == 0:
            print(f"📋 First Chunk Example:")
            print(f"   Chunk ID: {chunk['chunk_id']}")
            print(f"   Text preview: {chunk['chunk_text'][:150]}...")
            print(f"   Tokens: {chunk['chunk_length_tokens']}")
            print()

# Create DataFrame
chunks_df = pd.DataFrame(chunks_df_list)

print(f"📊 Chunks DataFrame:")
print(chunks_df.head(10))
print(f"\n   Shape: {chunks_df.shape}")
print(f"\n   Token distribution:")
print(chunks_df['chunk_tokens'].describe())

## Cell 9: Statistics Report

Detaylı istatistik raporu oluşturalım.

In [ ]:
# Detailed statistics
stats_report = {
    'timestamp': datetime.now().isoformat(),
    'input': {
        'total_records': total_records,
        'file': str(PROCESSED_DIR),
    },
    'configuration': {
        'chunk_size_tokens': chunker.chunk_size,
        'overlap_tokens': chunker.overlap_size,
        'step_tokens': chunker.step_size,
        'min_chunk_tokens': chunker.min_chunk_size,
    },
    'output': {
        'total_chunks': total_chunks,
        'total_tokens': total_tokens,
        'output_file': str(output_file),
        'output_size_mb': output_file.stat().st_size / (1024*1024),
    },
    'statistics': {
        'chunks_per_record_avg': round(total_chunks / total_records, 2),
        'tokens_per_chunk_avg': round(total_tokens / total_chunks, 1),
        'tokens_per_record_avg': round(total_tokens / total_records, 1),
        'min_chunk_tokens': chunks_df['chunk_tokens'].min(),
        'max_chunk_tokens': chunks_df['chunk_tokens'].max(),
        'median_chunk_tokens': chunks_df['chunk_tokens'].median(),
    },
    'processing': {
        'time_seconds': round(processing_time, 2),
        'records_per_second': round(total_records / processing_time, 1),
        'errors': error_count,
    }
}

# Print report
print("="*60)
print("CHUNKING STATISTICS REPORT")
print("="*60)
print(f"\n📥 INPUT:")
for k, v in stats_report['input'].items():
    print(f"   {k}: {v}")

print(f"\n⚙️  CONFIGURATION:")
for k, v in stats_report['configuration'].items():
    print(f"   {k}: {v}")

print(f"\n📤 OUTPUT:")
for k, v in stats_report['output'].items():
    if k != 'output_size_mb':
        print(f"   {k}: {v}")
    else:
        print(f"   {k}: {v:.2f}")

print(f"\n📊 STATISTICS:")
for k, v in stats_report['statistics'].items():
    print(f"   {k}: {v}")

print(f"\n⏱️  PERFORMANCE:")
for k, v in stats_report['processing'].items():
    print(f"   {k}: {v}")

print(f"\n" + "="*60)

## Cell 10: Save Statistics Report

Statistics report'unu JSON ve CSV olarak kaydet.

In [ ]:
# Save statistics as JSON
stats_json_file = CHUNKED_DIR / 'chunking_statistics.json'
with open(stats_json_file, 'w', encoding='utf-8') as f:
    json.dump(stats_report, f, indent=2, ensure_ascii=False)

print(f"✅ JSON report saved: {stats_json_file}")

# Save chunk summary as CSV
csv_file = CHUNKED_DIR / 'chunk_summary.csv'
chunks_df.to_csv(csv_file, index=False, encoding='utf-8')

print(f"✅ CSV summary saved: {csv_file}")

# List output files
print(f"\n📁 Output files in {CHUNKED_DIR}:")
for f in sorted(CHUNKED_DIR.glob('*')):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"   {f.name}: {size_mb:.2f} MB")